<a href="https://colab.research.google.com/github/satifyy/AI-For-Beginners/blob/main/Intro/assignment-lunar_lander.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Implement a Simple Reflex Agent

Name: Santiago Perafan Alvarez

SMU ID: 49713983


Learning Goals:
* How to install and use the Gymnasium environment.
* Identify components of the environment.
* Experiment with simple rules in a reinforcement learning setting.

AI tool usage:
* This is a simple exercise, you **cannot use AI.**

Instruction: Complete this notebook, run all cells, convert to HTML and upload to Canvas.

## Task 1: Setup [2 point]

Follow the steps in [Setup Gymnasium](../common/README.md) to install Gymnasium.

In [1]:
!apt-get -qq update
!apt-get -qq install -y swig xvfb ffmpeg
%pip install -q "gymnasium[box2d,classic-control,other]>=1.0,<2" pyvirtualdisplay
%pip install -q "gym-classics2 @ git+https://github.com/mhahsler/gym-classics2.git"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package swig4.0.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../swig4.0_4.0.2-1ubuntu1_amd64.deb ...
Unpacking swig4.0 (4.0.2-1ubuntu1) ...
Selecting previously unselected package swig.
Preparing to unpack .../swig_4.0.2-1ubuntu1_all.deb ...
Unpacking swig (4.0.2-1ubuntu1) ...
Setting up swig4.0 (4.0.2-1ubuntu1) ...
Setting up swig (4.0.2-1ubuntu1) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 65.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Prepa

## Task 2: Environment Components [2 Points]

You will implement a simple reflex-based agent for the Lunar Lander environment.

About the environment:
* Gymnasium documentation for the [Lunar Lander Environment](https://gymnasium.farama.org/environments/box2d/lunar_lander/)
* [Class example](lunar_lander.ipynb) with code for a simple agent.

Answer the following questions and explain your answers:

#### 1. Is the task episodic or continuous?

The task would be episodic because there is a clear ending whether that is a landing, crash, going off screen, or exceeding the step time limit (terminal states).

#### 2. Observability and Information States

Is the environment fully observable? Do the observations suffice to describe the current state of the environment?

Yes the environment is fully observable, this means the agent has access to know everything about the environment via its observations. The 8 dimensional observation vector captures all the relevant information that we need like coordinates, linear velocities, angle, angular velocity, and leg contacts. This suffices because there is no historical information needed to describe the current condition of the state.

#### 3. Continuous state space

Describe the factored representation of the state. Many fluents are continuous and not discrete. How does your agent deal with them?

The factored representation of the state means that the state isn't just state A or state B but rather a vector of distinct variables also known as fluents. For our lunar lander we have coordinates $x, y$, linear velocities $v_x, v_y$, angle $\theta$, and angular velocity $\omega$ as well as discrete boolean flags for leg ground contact. We deal with continuous fluents using thresholds instead of having unique if statements (which would be an endless amount of them) for example if the angle goes past past a certain amount it can correct rather than having an if for each angle past lets say 90.

#### 4. Policy

What is a policy? Why can the rules in a rule-based agent be seen as the definition of a policy? Does it define a stochastic or a deterministic policy?

A policy is essentially what action the agent takes based on the current state. In rule-based agents, the "rules" can be seen as policies because they perform a similar function in telling the agent what it should do given the state. It would map to a deterministic policy because, in a certain state, it will take the same action with 100% certainty rather than leave it up to the probabilities.

## Task 3: Implement A Better Reflex-Based Agent [6 Points]

Build a better that uses its right and left thrusters to land the craft (more) safely. Test your agent function using 100 random problems and use this feedback to improve your agent.

1. Add all the code that is needed to run your agent below.

In [4]:
import gymnasium as gym
import numpy as np
from enum import Enum
from gymnasium.wrappers import TransformReward

In [5]:
class Act(Enum):
    NO_OP = 0
    LEFT = 1
    MAIN = 2
    RIGHT = 3

class Obs(Enum):
    X = 0
    Y = 1
    VX = 2
    VY = 3
    ANGLE = 4
    ANGULAR_VELOCITY = 5
    LEFT_LEG_CONTACT = 6
    RIGHT_LEG_CONTACT = 7

In [6]:
def filter_reward(r):
    if r == 100.0 or r == -100.0:
        return r
    return 0.0

In [7]:
def run_episode(agent_function, env, max_steps=1000, verbose = True, render = True):
    """Run one episode in the environment using the provided agent."""

    # Reset the environment to generate the first observation (observation and state are the same in fully observable environments)
    observation, info = env.reset()

    # run one episode
    G = 0 # undiscounted episode return
    for i in range(max_steps):
        # call the agent function to select an action
        action = agent_function(observation)

        # step: execute an action in the environment
        observation_prime, reward, terminated, truncated, info = env.step(action)

        if verbose:
            print (f"Step {i+1}: Obs {np.round(observation, 1)} -> Action {action} - > Reward {np.round(reward,1)}, Obs' {np.round(observation_prime,1)}")

        observation = observation_prime
        G += reward

        # render the environment
        if render:
            env.render()

        if terminated:
            break

    if verbose:
        print(f"Episode Return: {G}")

    return G

In [32]:
def better_reflex_agent(observation):
    # state values
    angle = observation[Obs.ANGLE.value]
    angular_vel = observation[Obs.ANGULAR_VELOCITY.value]
    vy = observation[Obs.VY.value]
    vx = observation[Obs.VX.value]
    x = observation[Obs.X.value]

# predict target tilt based on current angle + angular speed
# Target angle: tilt slightly towards pad (x = 0), capped to avoid rolling over
    target_angle = np.clip((x * 0.4 + vx * 0.7), -0.3, 0.3)
    angle_error = angle - target_angle

    # 1. Primary: Orientation control with angular velocity damping
    if angle_error + 0.35 * angular_vel > 0.05:
        return Act.RIGHT.value
    elif angle_error + 0.35 * angular_vel < -0.05:
        return Act.LEFT.value

    # 2. Secondary: Fire main engine to cushion vertical descent
    target_vy = -0.3 if observation[Obs.Y.value] > 0.3 else -0.05
    if vy < target_vy:
        return Act.MAIN.value

    return Act.NO_OP.value

2. Add the code to perform 100 simulation runs and report the success rate.

In [21]:
def run_episodes(agent_function, env, n=100):
    """Run multiple episodes with the given agent and return the rewards for each episode."""

    Returns = []
    successes = []
    for _ in range(n):
        Return = run_episode(agent_function, env, verbose=False, render=False)
        Returns.append(Return)

    return Returns

In [33]:
# Create the environment with the reward wrapper
env = gym.make("LunarLander-v3")
env = TransformReward(env, filter_reward)

# Run evaluation with a fixed seed for reproducibility
env.reset(seed=42)
rng = np.random.default_rng(42)

Returns = run_episodes(better_reflex_agent, env, n=100)
env.close()

avg_return = np.average(Returns)
success_rate = np.average(np.array(Returns) == 100.0)

print(f"Average Returns: {avg_return}")
print(f"Success rate: {success_rate * 100:.1f}%")

Average Returns: 98.0
Success rate: 99.0%


In [27]:
# 1. Download the recorder helper script from the course repository
import urllib.request
import os

helper_url = "https://raw.githubusercontent.com/mhahsler/Introduction_to_Reinforcement_Learning/refs/heads/main/common/gymnasium_display_recorder.py"
if not os.path.exists("gymnasium_display_recorder.py"):
    urllib.request.urlretrieve(helper_url, "gymnasium_display_recorder.py")

from gymnasium_display_recorder import VideoWrapper, show

# 2. Wrap the environment with rgb_array rendering and the video recorder
env_vis = gym.make("LunarLander-v3", render_mode="rgb_array")
env_vis = TransformReward(env_vis, filter_reward)
env_vis_record = VideoWrapper(env_vis, "reflex_lander", render_fps=30)

# 3. Run an episode and display the recorded MP4 directly in the notebook
episode_return = run_episode(better_reflex_agent, env_vis_record, verbose=False)
print(f"Recorded Episode Return: {episode_return}")
show(env_vis_record)

Videos already exist, I remove them first!
Recorded Episode Return: 100.0
Showing: ./videos/video_reflex_lander-episode-0.mp4


3. How well does your agent work? Describe what the most important rules are that your agent uses.

The agent achieved a 71.0% landing success rate over 100 simulations with an average return of 62.0, improving upon the class example (which was around 0% to 1%). The first priority was to ensure stabilization, which was done by calculating a target tilt based on horizontal offset and drift velocity. It checked the angle error combined with angular velocity damping against a $\pm 0.02$ threshold to fire the left or right thrusters to prevent rolling over. The second rule managed descent velocity to ensure a soft landing; it checked descent velocity against an altitude-based threshold, allowing a higher velocity when the craft was farther from the ground and enforcing thruster engagement closer to the surface. The final rule was to do nothing (NO_OP) when conditions were within optimal tilt and descent thresholds.
Fine-tuning the values yielded an improved 99.0% landing success rate with an average return of 98.0. Widening the angle error bounds from $\pm 0.02$ to $\pm 0.05$ prevented over-controlling and gave the agent adequate tolerance to settle. Additionally, changing the target descent speed from $-0.01$ to $-0.05$ allowed the craft to touch down at a controlled pace rather than hovering nearly motionless. This maintains the original rule set while optimizing the decision boundaries.

4. Reinforcement learning will use feedback from interacting with the environment to learn a policy. A policy prescribes an action for each state. Describe how your rules can be seen as defining a policy in a discretized state space.

Well they can be defined as a policy because they give the agent instructions on what to do in certain states. When the tilt is too much it reaches the threshold the agent fires a thruster to correct its angle, or when its approaching the ground too quickly (exceeding a descent speed threshold) it will slow itself down to land gracefully. We also have to mention the do nothing instruction when we are at optimal tilt and descent speed. All three of these rules essentially assign a single action to each. This is the implementation of a deterministic policy over a discretized state space because we basically split up our infinite space into intervals using thresholds to make the problem finite.


&copy; 2025 [Michael Hahsler](https://michael.hahsler.net).
This work is openly licensed under [Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0) License](https://creativecommons.org/licenses/by-sa/4.0/)

![CC BY-SA 4.0](https://licensebuttons.net/l/by-sa/3.0/88x31.png)